In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

In [ ]:
# ── Part 1: Manual PCA (2D → 1D on simple XY data) ──────────────────────────
df = pd.DataFrame({'X': [2,4,6,8,10], 'Y': [1,3,5,7,9]})
print(df)

In [ ]:
# Step 1: Standardize
scaled = StandardScaler().fit_transform(df)

# Step 2: Covariance matrix
cov_matrix = np.cov(scaled.T)
print("Covariance Matrix:\n", cov_matrix)

# Step 3: Eigendecomposition → sort descending
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
idx          = np.argsort(eigenvalues)[::-1]
eigenvalues  = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]
print("Eigenvalues:", eigenvalues)
print("Eigenvectors:\n", eigenvectors)

# Step 4: Project to 1D
reduced = scaled @ eigenvectors[:, :1]
print("Reduced Data:\n", reduced)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].scatter(scaled[:, 0], scaled[:, 1])
axes[0].set(title="Original 2D Data", xlabel="X", ylabel="Y")

axes[1].scatter(reduced, np.zeros_like(reduced))
axes[1].set(title="Reduced to 1D via PCA", xlabel="PC1", ylabel="")

plt.tight_layout()
plt.show()

In [ ]:
# sklearn shortcut — same result
pca_1d = PCA(n_components=1)
print("Sklearn PCA output:\n", pca_1d.fit_transform(scaled))
print("Explained Variance Ratio:", pca_1d.explained_variance_ratio_)

In [ ]:
# ── Part 2: PCA with sklearn on Iris (4 features → 2 PCs) ───────────────────
iris = load_iris()
X, y = iris.data, iris.target
print("Shape:", X.shape)   # (150, 4)

X_scaled = StandardScaler().fit_transform(X)

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print("Explained Variance Ratio:", pca.explained_variance_ratio_)
print(f"Total Variance Explained: {sum(pca.explained_variance_ratio_)*100:.2f}%")
print("Principal Components:\n", pca.components_)

In [ ]:
for i, name in enumerate(iris.target_names):
    plt.scatter(X_pca[y == i, 0], X_pca[y == i, 1], label=name)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA on Iris Dataset")
plt.legend()
plt.show()

In [ ]:
# LR on raw (4D) vs PCA (2D) — compare accuracy
X_tr,  X_te,  y_tr, y_te = train_test_split(X,     y, test_size=0.2, random_state=42)
Xp_tr, Xp_te, _,    _    = train_test_split(X_pca, y, test_size=0.2, random_state=42)

lr_raw = LogisticRegression(max_iter=1000).fit(X_tr,  y_tr)
lr_pca = LogisticRegression(max_iter=1000).fit(Xp_tr, y_tr)

print(f"LR on raw features (4D): {lr_raw.score(X_te,  y_te):.4f}")
print(f"LR on PCA features (2D): {lr_pca.score(Xp_te, y_te):.4f}")